##  correct week - 4

## 1. IMPORT LIBRARIES

In [1]:
import pandas as pd
from datetime import datetime

## 2. LOGGING FUNCTION

In [2]:
def log(message):
    print(f"[{datetime.now()}] {message}")

## 3. LOAD DATA

In [3]:
def load_data(path):
    df = pd.read_csv(path, encoding='ISO-8859-1')
    
    # Clean column names (remove extra spaces)
    df.columns = df.columns.str.strip()

    log(f"Columns found: {df.columns.tolist()}")
    log(f"Data loaded. Shape: {df.shape}")
    return df

## 4. DATA CLEANING

In [4]:
def clean_data(df):
    log("Cleaning data...")

    # Handle column name variations
    if "Customer ID" in df.columns:
        df = df.dropna(subset=["Customer ID"]).copy()
        cust_col = "Customer ID"
    elif "CustomerID" in df.columns:
        df = df.dropna(subset=["CustomerID"]).copy()
        cust_col = "CustomerID"
    else:
        raise Exception("Customer column not found!")

    # Convert date
    df.loc[:, "InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

    # Remove invalid transactions (IMPORTANT FIX)
    df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

    # Create TotalAmount
    df.loc[:, "TotalAmount"] = df["Quantity"] * df["Price"]

    log(f"After cleaning: {df.shape}")
    return df, cust_col

## 5. CREATE RFM TABLE

In [5]:
def create_rfm(df, cust_col):
    log("Creating RFM table...")

    snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

    rfm = df.groupby(cust_col).agg({
        "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
        "Invoice": "nunique",
        "TotalAmount": "sum"
    })

    rfm.columns = ["Recency", "Frequency", "Monetary"]
    rfm = rfm.reset_index()

    log(f"RFM created. Shape: {rfm.shape}")
    return rfm

## 6. RFM SCORING

In [6]:
def rfm_scoring(rfm):
    log("Applying RFM scoring...")

    rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1])
    rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
    rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5])

    rfm["RFM_Score"] = (
        rfm["R_Score"].astype(str) +
        rfm["F_Score"].astype(str) +
        rfm["M_Score"].astype(str)
    )

    return rfm

## 7. CUSTOMER SEGMENTATION

In [7]:
def segment_customers(rfm):
    log("Segmenting customers...")

    def segment(row):
        if row["RFM_Score"] >= "444":
            return "High Value Customers"
        elif row["RFM_Score"] >= "333":
            return "Frequent Customers"
        else:
            return "Low Value Customers"

    rfm["Segment"] = rfm.apply(segment, axis=1)

    return rfm

## 8. SAVE OUTPUT

In [8]:
def save_output(rfm, path):
    rfm.columns = [col.replace(" ", "_") for col in rfm.columns]
    rfm.to_csv(path, index=False)

    log(f"File saved at: {path}")

## 9. MAIN PIPELINE FUNCTION

In [9]:
def run_pipeline():

    log("Pipeline started")

    file_path = r"C:\Users\Pravith Kumar J\Downloads\archive (9)\online_retail_II.csv"
    output_path = r"C:\Users\Pravith Kumar J\rfm_data_new.csv"

    df = load_data(file_path)

    df, cust_col = clean_data(df)

    rfm = create_rfm(df, cust_col)

    rfm = rfm_scoring(rfm)

    rfm = segment_customers(rfm)

    save_output(rfm, output_path)

    log("Pipeline executed successfully")

## 10. RUN PIPELINE

In [10]:
run_pipeline()

[2026-05-03 17:41:05.646080] Pipeline started
[2026-05-03 17:41:08.196991] Columns found: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
[2026-05-03 17:41:08.197614] Data loaded. Shape: (1067371, 8)
[2026-05-03 17:41:08.197650] Cleaning data...
[2026-05-03 17:41:13.186704] After cleaning: (805549, 9)
[2026-05-03 17:41:13.222869] Creating RFM table...
[2026-05-03 17:41:14.868832] RFM created. Shape: (5878, 4)
[2026-05-03 17:41:14.869562] Applying RFM scoring...
[2026-05-03 17:41:14.894472] Segmenting customers...
[2026-05-03 17:41:15.089930] File saved at: C:\Users\Pravith Kumar J\rfm_data_new.csv
[2026-05-03 17:41:15.090510] Pipeline executed successfully


In [11]:
 rfm = pd.read_csv(r"C:\Users\Pravith Kumar J\rfm_data_new.csv" )
print(rfm.shape)
print(rfm.head())

(5878, 9)
   Customer_ID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  \
0      12346.0      326         12  77556.46        2        5        5   
1      12347.0        2          8   5633.32        5        4        5   
2      12348.0       75          5   2019.40        3        4        4   
3      12349.0       19          4   4428.69        5        3        5   
4      12350.0      310          1    334.40        2        1        2   

   RFM_Score               Segment  
0        255   Low Value Customers  
1        545  High Value Customers  
2        344    Frequent Customers  
3        535  High Value Customers  
4        212   Low Value Customers  
